# Qixia ROCm LoRA Train

直接使用 LLaMA Factory 在 ROCm AMD GPU 上做一次 LoRA 微调。这个 notebook 不需要额外的大模型 API key。

默认数据：

- `data/qixia_train.json`: 6577 条完整已提取 ShareGPT 训练数据
- `data/qixia_valid.json`: 346 条验证数据
- `data/dataset_info.json`: LLaMA Factory 数据集配置


## 0. 参数区

先保持默认值跑通。确认数据、模型、显存都正常后，再把 `RUN_TRAIN` 改成 `True`。


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_repo_dir(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'configs').is_dir() and (path / 'data').is_dir() and (path / 'scripts').is_dir():
            return path
    raise RuntimeError(f'Could not find repo root from {start}')

REPO_DIR = find_repo_dir(Path.cwd().resolve())
PERSIST_DIR = REPO_DIR / 'network-workspace'
MODEL_IDS = {
    'qwen3_5_9b': 'Qwen/Qwen3.5-9B',
    'qwen3_6_35b_a3b': 'Qwen/Qwen3.6-35B-A3B',
}
MODEL_CHOICE = 'qwen3_5_9b'  # 可改为 'qwen3_6_35b_a3b'
MODEL_ID = MODEL_IDS[MODEL_CHOICE]
RUN_NAME = f'{MODEL_CHOICE}_lora'
CONFIG_PATH = REPO_DIR / 'configs' / f'{RUN_NAME}.yaml'
DATA_DIR = REPO_DIR / 'data'
OUTPUT_DIR = PERSIST_DIR / 'outputs' / RUN_NAME
LOG_DIR = PERSIST_DIR / 'logs'
MODEL_CACHE_DIR = PERSIST_DIR / 'models'
RUNTIME_CONFIG_DIR = PERSIST_DIR / 'runtime_configs'
RUNTIME_CONFIG_PATH = RUNTIME_CONFIG_DIR / f'{RUN_NAME}.yaml'

RUN_INSTALL = True
RUN_TRAIN = False

assert CONFIG_PATH.exists(), f'missing config: {CONFIG_PATH}'
assert DATA_DIR.exists(), f'missing data dir: {DATA_DIR}'

for path in [PERSIST_DIR, OUTPUT_DIR, LOG_DIR, MODEL_CACHE_DIR, RUNTIME_CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('python:', sys.version)
print('repo:', REPO_DIR)
print('persist:', PERSIST_DIR)
print('model_choice:', MODEL_CHOICE)
print('model_id:', MODEL_ID)
print('config:', CONFIG_PATH)
print('runtime_config:', RUNTIME_CONFIG_PATH)
print('data:', DATA_DIR)
print('model_cache:', MODEL_CACHE_DIR)
print('output:', OUTPUT_DIR)


## 1. 命令工具


In [ ]:
def run(cmd, cwd=REPO_DIR, timeout=None):
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}: {cmd}')
    return result


## 2. ROCm / PyTorch 自检

必须看到：

- `torch.version.hip` 有版本号
- `torch.cuda.is_available()` 是 `True`
- GPU 显存符合预期


In [ ]:
run('rocm-smi --showproductname --showmeminfo vram --showuse || true', timeout=60)

import torch
print('torch:', torch.__version__)
print('hip:', getattr(torch.version, 'hip', None))
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
assert getattr(torch.version, 'hip', None), '当前不是 ROCm/HIP 版 PyTorch，先不要继续。'
assert torch.cuda.is_available(), 'PyTorch 没有识别到 AMD GPU。'

props = torch.cuda.get_device_properties(0)
print('device:', torch.cuda.get_device_name(0))
print('total_memory_gb:', round(props.total_memory / 1024**3, 2))
print('bf16 supported:', torch.cuda.is_bf16_supported())


## 3. 安装训练依赖

`requirements-rocm.txt` 不包含 `torch`，避免覆盖云镜像已有的 ROCm PyTorch。


In [ ]:
if RUN_INSTALL:
    run('python -m pip install -U pip setuptools wheel', timeout=300)
    run('python -m pip install -r requirements-rocm.txt --upgrade-strategy only-if-needed', timeout=1200)
else:
    print('skip install')

import torch
print('torch after install:', torch.__version__)
print('hip after install:', getattr(torch.version, 'hip', None))
assert getattr(torch.version, 'hip', None), '安装依赖后 torch 不再是 ROCm 版。'


## 4. 数据校验

这里读的是仓库内完整数据，不是 sample。


In [ ]:
run('python scripts/validate_dataset.py data/qixia_train.json data/qixia_valid.json')

train = json.loads((DATA_DIR / 'qixia_train.json').read_text(encoding='utf-8'))
valid = json.loads((DATA_DIR / 'qixia_valid.json').read_text(encoding='utf-8'))
print('train examples:', len(train))
print('valid examples:', len(valid))
print(json.dumps(train[0], ensure_ascii=False, indent=2)[:1200])


## 5. 训练配置检查


In [ ]:
def replace_yaml_value(text: str, key: str, value: Path | str) -> str:
    value = str(value)
    lines = []
    replaced = False
    for line in text.splitlines():
        if line.startswith(f'{key}:'):
            lines.append(f'{key}: {value}')
            replaced = True
        else:
            lines.append(line)
    if not replaced:
        raise RuntimeError(f'missing yaml key: {key}')
    return '\n'.join(lines) + '\n'

assert CONFIG_PATH.exists()
assert (DATA_DIR / 'dataset_info.json').exists()
assert (DATA_DIR / 'qixia_train.json').exists()
assert (DATA_DIR / 'qixia_valid.json').exists()

print('downloading/checking local model from ModelScope:', MODEL_ID)
from modelscope import snapshot_download
MODEL_DIR = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE_DIR))).resolve()
assert (MODEL_DIR / 'config.json').exists(), f'missing model config.json: {MODEL_DIR}'

runtime_config = CONFIG_PATH.read_text(encoding='utf-8')
runtime_config = replace_yaml_value(runtime_config, 'model_name_or_path', MODEL_DIR)
runtime_config = replace_yaml_value(runtime_config, 'output_dir', OUTPUT_DIR)
runtime_config = replace_yaml_value(runtime_config, 'logging_dir', LOG_DIR)
RUNTIME_CONFIG_PATH.write_text(runtime_config, encoding='utf-8')

print('model_dir:', MODEL_DIR)
print('runtime config:')
print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))


## 6. 开始 LoRA 训练

确认前面都通过后，把参数区的 `RUN_TRAIN = True` 再运行这一格。


In [ ]:
if RUN_TRAIN:
    env = os.environ.copy()
    env['HIP_VISIBLE_DEVICES'] = '0'
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['PYTORCH_HIP_ALLOC_CONF'] = 'expandable_segments:True'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    run(f'python scripts/train_lora.py --config {RUNTIME_CONFIG_PATH}', timeout=None)
else:
    print('skip train. Set RUN_TRAIN=True after checking ROCm, deps, data, and config.')


## 7. 快速推理

训练完成后，把 `--adapter` 指向 LoRA 输出目录。


In [ ]:
adapter = OUTPUT_DIR
base_model = globals().get('MODEL_DIR', MODEL_ID)
print('adapter:', adapter)
print('base_model:', base_model)
print('run after training:')
print(f'python scripts/quick_infer.py --model {base_model} --adapter {adapter} --prompt "如果一个规则看起来互相矛盾，你会怎么判断？"')


## 8. 可选合并 LoRA

如果需要导出合并后的完整模型：


In [ ]:
merged_out = PERSIST_DIR / 'outputs' / f'{MODEL_CHOICE}_merged'
base_model = globals().get('MODEL_DIR', MODEL_ID)
print(f'python scripts/merge_lora.py --model {base_model} --adapter {OUTPUT_DIR} --out {merged_out}')
